## 🧠 YOLO-MultiSpectral: Demo Notebook UNDER CONSTRUCTION (30/6/2025)

### 🔍 Section 1: Introduction

This notebook demonstrates how to modify a YOLO model 5 channel input.
Dataset weeds galore


---

### ⚙️ Step 1: Setup Environment

Below we install dependencies and clone the YOLO-MultiSpec repository.


In [ ]:
!ls /content

In [ ]:
import shutil
import os


folder = "/content"

# 🔥 CAREFUL: This will delete EVERYTHING under /content
for item in os.listdir(folder):
    item_path = os.path.join(folder, item)
    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)  # remove file or symlink
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)  # remove directory
        print(f"✅ Deleted: {item_path}")
    except Exception as e:
        print(f"❌ Failed to delete {item_path}: {e}")

In [1]:
!python --version

Python 3.10.18


In [ ]:
import os
#  Clone repository -- 
#  Only required for running from COLAB
#run_from_colab = False



import os
import shutil
import subprocess


clone_dir = "YOLO-Multispectral"
# Delete if exists
if os.path.isdir(clone_dir):
    print(f"🧹 Deleting existing {clone_dir}")
    shutil.rmtree(clone_dir)


# Clone your GitHub repo
!git clone https://github.com/aesparon/YOLO-Multispectral.git

# Change directory to the repo root   REMOVE FOR COLAB
%cd YOLO-Multispectral

# Now install requirements
!pip install -r code/requirements.txt




In [9]:

# Check cuda support
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


cuda


### 🔍 Step 2. download weeds galore dataset

Download dataset

In [ ]:
import os
current_path = os.getcwd()


# add code path for script files
import os
import sys 
# yolo modified source folder
yolo_source_path = os.path.abspath(current_path + "/code/ultralytics_MS/")
sys.path.insert(0, yolo_source_path)
import ultralytics

# code folder
yolo_source_path = os.path.abspath(current_path +  "/code/")
sys.path.insert(0, yolo_source_path)

In [ ]:
from utils_downloader import download_and_extract_zip

# Set parameters
url = "https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip"
output_zip = current_path + "/datasets/weedsgalore-dataset.zip"
extract_dir = current_path + "/datasets/"

# Call the function
download_and_extract_zip(url, output_zip, extract_dir)

🔍 Checking file info...
📥 Downloading with resume support...


Downloading: 100%|██████████| 321M/321M [02:24<00:00, 2.33MB/s] 


✅ Download complete: weedsgalore-dataset.zip
📦 Extracting...


Extracting: 100%|██████████| 1119/1119 [00:01<00:00, 844.27file/s]

✅ Extracted to: weedsgalore-dataset


### 🔍 Section 3: Create train/val/test


In [ ]:
from pre_process_images import *

# Step 1.  Merge all date folder images into single folder  ###########################################################
input_folders = [
    extract_dir + "/weedsgalore-dataset/2023-05-25/images/",
    extract_dir + "/weedsgalore-dataset/2023-05-30/images/",
    extract_dir + "/weedsgalore-dataset/2023-06-06/images/",
    extract_dir + "/weedsgalore-dataset/2023-06-15/images/"
]
images_merge_band_images = extract_dir + "/weedsgalore-dataset/images_merge/"

# merge all band images from all date folders into a single folder
merge_files(input_folders, images_merge_band_images)

# Step 2.  Stack RGB and RGBRN images and convert from uint16 to uint8 and split train/val/test  #######################################################3


#data_type = 'uint8'    # 'uint16'     #  yolo does not train on uint16 but will mod yolo to test any increase in accuracy using uinr16 over uint8 (future wotk)
uint16_to_uint8_method = 'stretch'    # 'normalize' 

# stack RGB and convert to uint8 and separate to train/val/test using same split as weeds-galore
output_RGB_train_val_test_path = stack_RGB_set_uint8_sort_train_val_test (images_merge_band_images , uint16_to_uint8_method )

# stack RGBRN and convert to uint8 and separate to train/val/test using same split as weeds-galore
output_RGBRN_train_val_test_path =  stack_RGBRN_set_uint8_sort_train_val_test (images_merge_band_images , uint16_to_uint8_method )



✅ Copied: ./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-05-25_0109_B.png → ./weedsgalore-dataset/images_merge/2023-05-25_0109_B_9.png
✅ Copied: ./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-05-25_0109_G.png → ./weedsgalore-dataset/images_merge/2023-05-25_0109_G_9.png
✅ Copied: ./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-05-25_0109_NIR.png → ./weedsgalore-dataset/images_merge/2023-05-25_0109_NIR_9.png
✅ Copied: ./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-05-25_0109_R.png → ./weedsgalore-dataset/images_merge/2023-05-25_0109_R_9.png
✅ Copied: ./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-05-25_0109_RE.png → ./weedsgalore-dataset/images_merge/2023-05-25_0109_RE_9.png
✅ Copied: ./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-05-25_0114_B.png → ./weedsgalore-dataset/images_merge/2023-05-25_0114_B_9.png
✅ Copied: ./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-0

c:\Users\andre\.conda\envs\yolo-ms3\lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
c:\Users\andre\.conda\envs\yolo-ms3\lib\site-packages\rasterio\__init__.py:378: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0119.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0147.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0150.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0158.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0170.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0183.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0189.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0219.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0222.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0228.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0363.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0379.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0383.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0402.tif
✅ Saved: ./weedsgalore-dataset/RGB/RGB_uint16/2023-05-25_0406.tif
✅ Saved: .

In [3]:

output_RGBRN_train_val_test_path


'./weedsgalore-dataset/train_val_test/RGBRN/'

### Train model paramters

In [3]:
from pathlib import Path

# add hic fix - test and fix - then remove
#os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


train_data = 'weed_RGBRN' 

# parameters
num_epochs = 1000   # force stop by patience
batch_size = 16   # 4   # yolo_config['batch']
patience = 50
images_test_path = ''    # set if using test

#data_base_path =  'D:/PD/yolo_mod/data/WeedsGalore/data_processed/train_val_test_V7/RGBNR/'
project_base_path =  output_RGBRN_train_val_test_path + '/outputs/'      #  'D:/PD/yolo_mod/working2/projects/weeds_galore/RGBRN_tests_final2/'
os.makedirs(project_base_path, exist_ok=True)

##############################################   parameters   ##########################################################
# can auto detect
number_of_channels = 5   
clip_size= "600"
# ############################      uint16      ###################################################################
# ############################      uint8      ###################################################################
# expect folders for images/labels in train/val/test
images_train_path= output_RGBRN_train_val_test_path +  "train/" 
images_val_path=  output_RGBRN_train_val_test_path + "val/" 
# set to '' or None id NO test data
images_test_path=  output_RGBRN_train_val_test_path + "test/"     

classes_list = ['maize','amaranth','grass','quickweed','other']
number_of_classes = len(classes_list)

model_base =   'yolo11x-seg.yaml'    # 'yolo11n-seg.pt'
folder = model_base[0:6]
#Initialise additional channels when more then 3 (RGB)
channel_init_mode = 'random'       # 'avg'   'zeros'     'random'    'same'
# trans_learn = True     # False      # True       #False    #True

use_cbam=False
use_eca=False
use_spectral=False 
use_dropblock=False
drop_prob=0.1

In [4]:
# create train yaml
def create_train_yaml(images_train_path, images_val_path ,classes_list , number_of_channels, output_yaml ,  images_test_path = None):
    ###    CREATE data - YAML file for training    #######################################################################
    yaml_content = "train: " + images_train_path + "\n"
    yaml_content =yaml_content +  "val: " + images_val_path + "\n"
    if not (images_test_path == '' or images_test_path == None):
        yaml_content =yaml_content +  "test: " + images_test_path + "\n"

    # extra channels
    yaml_content =yaml_content +  "nc: " + str ( len(classes_list) ) + "\n"
    yaml_content =yaml_content +  "channels: " + str( number_of_channels) + "\n"
    #yaml_content =yaml_content  + cat_yaml_str     #"names: ['tree']"

    #classes_list = ['maize','amaranth','grass','quickweed','other']  
    # create name classes for yolo training
    names_str = "names: [" + ",".join(classes_list) + "]"
    print(names_str)
    yaml_content =yaml_content +  names_str + "\n"

    #output_yaml =  project_base_path + "data_" + train_data + ".yaml"
    

    with Path(output_yaml).open('w') as f:
        f.write(yaml_content) 
    ##########################################################################################################################



In [5]:
output_yaml =  project_base_path + "data_" + train_data + ".yaml"
    
create_train_yaml(images_train_path, images_val_path ,classes_list , number_of_channels, output_yaml ,  images_test_path = None)

names: [maize,amaranth,grass,quickweed,other]


In [6]:
from mod_pt_model_seg import *

model_base_str = model_base.replace('.','_') # for file path - remove .extension
output_model_train_base =  project_base_path  +  train_data + '_' + model_base_str + '_method_' + channel_init_mode + '_eps_' + str(num_epochs) 
#os.makedirs(output_model_train_base, exist_ok=True)

# from mod_pt_model_seg_v1.py
model_created ,output_model_train  = patch_yolo_seg_ckpt (
    #input_ckpt='C:/Users/andre/AppData/Roaming/QGIS/QGIS3/profiles/default/python/plugins/aigist/QGIS_DL/models_base/YOLO8/yolo11x-seg.pt',
    model_base=model_base,
    output_model_train= output_model_train_base,
    in_channels=number_of_channels,
    nc=number_of_classes,
    #trans_learn = trans_learn,
    channel_init_mode=channel_init_mode,
    use_cbam=use_cbam, 
    use_eca=use_eca, 
    use_spectral=use_spectral, 
    use_dropblock=use_dropblock, 
    drop_prob=0.1
)


📦 Loading pretrained YOLO segmentation model: yolo11x-seg.yaml
🛠️ Patching input conv from 3 to 5 using 'random'
🧠 Updating segmentation head to 5 classes
Saving patched model to: ./weedsgalore-dataset/train_val_test/RGBRN//outputs/weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg/mod_model/yolo11x-seg_no_TL.pt


In [10]:
output_yaml = 'D:/PD/Publications/Yolo_mod/github/colab_test/notebooks/weedsgalore-dataset/train_val_test/RGBRN/outputs/data_weed_RGBRN.yaml'
print(model_file)


yolo11x-seg_no_TL.pt


In [9]:
import sys
import os

yolo_source_path = 'D:/PD/yolo_mod/working2/yolo_source/yolo_2025_06_04_mod/ultralytics-main/'
sys.path.insert(1, yolo_source_path)
import ultralytics
from ultralytics import YOLO
#from pathlib import Path

print('here')
# output_dir = Path("runs/segment/weedsgalore-dataset/train_val_test/RGBRN/outputs/weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg")
# output_dir.mkdir(parents=True, exist_ok=True)

# print("📂 Output path:", output_dir.resolve())

# use created model   #################################################
model_path =os.path.dirname(model_created)
model_file = os.path.basename(model_created)

#  Check later - cannot pass path to   -> model = YOLO( path + 'yolo11x.pt') 
if model_path != '':
    #os.chdir(model_path)
    os.chdir('D:/PD/Publications/Yolo_mod/github/colab_test/notebooks/weedsgalore-dataset/train_val_test/RGBRN/outputs/weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg/mod_model/')
    
model = YOLO( model_file )


# Automatically use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

##########################################  TRAIN  #######################################
results = model.train(
        #task='segment',
        batch= batch_size,   # batch_size,    # 32 on small    set to -1 to optimise to GPU
        data=output_yaml,
        epochs= num_epochs ,      #num_epochs,
        imgsz=int(clip_size),
        device=device,
        patience =patience ,
        #conf = conf,
        name = 'RGBRN_y11x',   # 'D:/PD/Publications/Yolo_mod/github/colab_test/notebooks/weedsgalore-dataset/train_val_test/RGBRN/outputs/weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg/' ,   #output_model_train,
        exist_ok=True,
)  


here
New https://pypi.org/project/ultralytics/8.3.160 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.148  Python-3.10.18 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/PD/Publications/Yolo_mod/github/colab_test/notebooks/weedsgalore-dataset/train_val_test/RGBRN/outputs/data_weed_RGBRN.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1000, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=600, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yo

RuntimeError: Dataset 'D://PD/Publications/Yolo_mod/github/colab_test/notebooks/weedsgalore-dataset/train_val_test/RGBRN/outputs/data_weed_RGBRN.yaml' error  Dataset 'D://PD/Publications/Yolo_mod/github/colab_test/notebooks/weedsgalore-dataset/train_val_test/RGBRN/outputs/data_weed_RGBRN.yaml' images not found, missing path 'D:\PD\Publications\Yolo_mod\github\colab_test\notebooks\weedsgalore-dataset\train_val_test\RGBRN\outputs\weedsgalore-dataset\train_val_test\RGBRN\val'
Note dataset download directory is 'D:\PD\yolo_mod\working2\test_mod_in_channels\datasets'. You can update this in 'C:\Users\andre\AppData\Roaming\Ultralytics\settings.json'

In [26]:
results.save_dir

WindowsPath('D:/PD/yolo_mod/working2/projects/weeds_galore/RGBRN_tests_final2/weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg')

In [32]:

try:
    # can be incremented on 2nd run
    output_model_train =results.save_dir
except NameError:
    output_model_train = output_model_train

if images_test_path != '':
    # run test evaluation if exists
    model_save_path = str(output_model_train)
    best_last = 'best'
    eval_best_last(model_save_path , output_yaml , best_last)

    best_last = 'last'
    eval_best_last(model_save_path , output_yaml , best_last)

Ultralytics 8.3.148  Python-3.10.18 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
YOLO11x-seg summary (fused): 203 layers, 62,009,631 parameters, 0 gradients, 318.9 GFLOPs


FileNotFoundError: [34m[1mval: [0mError loading data from None
See https://docs.ultralytics.com/datasets for dataset formatting guidance.